# Actor Clustering - Girvan-Newman

This notebook implements the actor-level clustering pipeline for one model using the shared preprocessing and graph construction logic.


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.clustering.girvan_newman import run_girvan_newman
from src.preprocessing.pipeline import prepare_common_actor_data
from src.utils.config import RESULTS_DIR
from src.utils.io_utils import save_json, save_results_csv

MAX_NODES = 256
N_CLUSTERS = 4
common = prepare_common_actor_data(max_nodes=MAX_NODES)
assignments, metrics = run_girvan_newman(common["graph"], n_clusters=N_CLUSTERS)
result_dir = RESULTS_DIR / "girvan_newman"
save_results_csv(assignments, result_dir / "cluster_assignments.csv")
save_json(metrics, result_dir / "metrics.json")
print(metrics)

{'method': 'girvan_newman', 'n_nodes': 256, 'n_edges': 152, 'n_clusters': 189, 'modularity': 0.7349056318862632, 'runtime_seconds': 0.24944343000242952, 'requested_clusters': 4}


In [2]:
from src.clustering.evaluation import evaluate_clusters
from src.clustering.visualization import save_cluster_figures, save_graph_partition_figure

labels = common["actor_df"].set_index("actor_id")["class"].reindex(assignments["actor_id"])
metrics.update(evaluate_clusters(assignments, labels=labels))
metrics["research_questions"] = [
    "Can edge-betweenness removal expose bridge addresses in a reproducible small-graph benchmark?",
    "Does its partition differ materially from modularity-based methods?",
]
save_json(metrics, result_dir / "metrics.json")
save_cluster_figures(assignments, result_dir, title="Girvan-Newman actor communities")
save_graph_partition_figure(common["graph"], assignments, result_dir / "graph_partition.png", "Girvan-Newman partition on benchmark graph")
print("Research interpretation metrics:", {key: metrics[key] for key in ("modularity", "ari", "nmi", "ami")})

Research interpretation metrics: {'modularity': 0.7349056318862632, 'ari': 0.0003293820111989884, 'nmi': 0.01444831983350512, 'ami': 0.0021792787055130787}
